In [1]:
# import the proper packages
import pandas as pd
import gurobipy as gp
from gurobipy import GRB

## Gourmet Gizmos

Assume that 3 plant locations are up and running so you don't have to worry about fixed costs.

In [2]:
# Start with Gourmet Gizmos
# Get data
# Gourmet Gizmos demand is in a sheet called 'gg_demand'
gg_demand = pd.read_excel('KitchenCompanies.xlsx', sheet_name='gg_demand', index_col=0)
gg_demand

,demand
city,
Atlanta,10
Boston,8
Chicago,14


In [3]:
# Convert demand to a dictionary
gg_demand_as_dict = gg_demand['demand'].to_dict()
gg_demand_as_dict

{'Atlanta': 10, 'Boston': 8, 'Chicago': 14}

In [4]:
# Create the markets as a list
gg_markets = gg_demand.index.to_list()
gg_markets

['Atlanta', 'Boston', 'Chicago']

In [5]:
# Get data
# the capacity, variable, and fixed costs are in a sheet labeled 'gg_costs_and_cap'
gg_cost_and_cap = pd.read_excel('KitchenCompanies.xlsx', sheet_name='gg_costs_and_cap',
                                index_col=0)
gg_cost_and_cap

,Atlanta,Boston,Chicago,monthly_capacity,monthly_fc
supply_city,,,,,
Bethesda,1675,400,685,18,7650
Memphis,380,1355,543,22,4100
Kansas_City,922,1646,700,31,2200


In [6]:
# Create the plants as a list
gg_plants = gg_cost_and_cap.index.to_list()
gg_plants

['Bethesda', 'Memphis', 'Kansas_City']

In [7]:
# Create a dictionary to hold the capacity for each plant
gg_capacity = gg_cost_and_cap['monthly_capacity'].to_dict()
gg_capacity

{'Bethesda': 18, 'Memphis': 22, 'Kansas_City': 31}

In [ ]:
# Create a dictionary to hold the fixed cost for each plant


In [8]:
# Get data
# Transportation costs are in our gg_costs_and_cap DataFrame
gg_var_cost = gg_cost_and_cap[gg_markets]
gg_var_cost

,Atlanta,Boston,Chicago
supply_city,,,
Bethesda,1675,400,685
Memphis,380,1355,543
Kansas_City,922,1646,700


In [10]:
# Convert costs to a list of lists
gg_transportation_costs = gg_var_cost.values.tolist()
gg_transportation_costs

[[1675, 400, 685], [380, 1355, 543], [922, 1646, 700]]

In [11]:
# Create the model instance and set to minimize
gg = gp.Model('Gourmet_Gizmos')
gg.ModelSense = GRB.MINIMIZE

Restricted license - for non-production use only - expires 2026-11-23


In [12]:
# Create the flow variables from plants to markets
flow = gg.addVars(gg_plants, gg_markets, obj=gg_transportation_costs, name='flow')

# Update the model and display
gg.update()
gg.display()

Minimize
1675.0 flow[Bethesda,Atlanta] + 400.0 flow[Bethesda,Boston]
+ 685.0 flow[Bethesda,Chicago] + 380.0 flow[Memphis,Atlanta]
+ 1355.0 flow[Memphis,Boston] + 543.0 flow[Memphis,Chicago]
+ 922.0 flow[Kansas_City,Atlanta] + 1646.0 flow[Kansas_City,Boston]
+ 700.0 flow[Kansas_City,Chicago]
Subject To


/tmp/ipykernel_535/551310936.py:6: DeprecationWarning: Model.display() is deprecated
  gg.display()


In [13]:
# Make sure you meet demand for each market
for j in gg_markets:
    gg.addConstr(flow.sum('*', j) >= gg_demand_as_dict[j], name=f'demand_{j}')

# Update and display
gg.update()
gg.display()

Minimize
1675.0 flow[Bethesda,Atlanta] + 400.0 flow[Bethesda,Boston]
+ 685.0 flow[Bethesda,Chicago] + 380.0 flow[Memphis,Atlanta]
+ 1355.0 flow[Memphis,Boston] + 543.0 flow[Memphis,Chicago]
+ 922.0 flow[Kansas_City,Atlanta] + 1646.0 flow[Kansas_City,Boston]
+ 700.0 flow[Kansas_City,Chicago]
Subject To
demand_Atlanta: flow[Bethesda,Atlanta] + flow[Memphis,Atlanta] +
 flow[Kansas_City,Atlanta] >= 10
demand_Boston: flow[Bethesda,Boston] + flow[Memphis,Boston] + flow[Kansas_City,Boston]
 >= 8
demand_Chicago: flow[Bethesda,Chicago] + flow[Memphis,Chicago] +
 flow[Kansas_City,Chicago] >= 14


/tmp/ipykernel_535/3129907510.py:7: DeprecationWarning: Model.display() is deprecated
  gg.display()


In [14]:
# Make sure stay under copacity at plants
for i in gg_plants:
    gg.addConstr(flow.sum(i, '*') <= gg_capacity[i], name=f'capacity_{i}')

# Update and display
gg.update()
gg.display()

Minimize
1675.0 flow[Bethesda,Atlanta] + 400.0 flow[Bethesda,Boston]
+ 685.0 flow[Bethesda,Chicago] + 380.0 flow[Memphis,Atlanta]
+ 1355.0 flow[Memphis,Boston] + 543.0 flow[Memphis,Chicago]
+ 922.0 flow[Kansas_City,Atlanta] + 1646.0 flow[Kansas_City,Boston]
+ 700.0 flow[Kansas_City,Chicago]
Subject To
demand_Atlanta: flow[Bethesda,Atlanta] + flow[Memphis,Atlanta] +
 flow[Kansas_City,Atlanta] >= 10
demand_Boston: flow[Bethesda,Boston] + flow[Memphis,Boston] + flow[Kansas_City,Boston]
 >= 8
demand_Chicago: flow[Bethesda,Chicago] + flow[Memphis,Chicago] +
 flow[Kansas_City,Chicago] >= 14
capacity_Bethesda: flow[Bethesda,Atlanta] + flow[Bethesda,Boston] +
 flow[Bethesda,Chicago] <= 18
capacity_Memphis: flow[Memphis,Atlanta] + flow[Memphis,Boston] + flow[Memphis,Chicago]
 <= 22
capacity_Kansas_City: flow[Kansas_City,Atlanta] + flow[Kansas_City,Boston] +
 flow[Kansas_City,Chicago] <= 31


/tmp/ipykernel_535/4155635282.py:7: DeprecationWarning: Model.display() is deprecated
  gg.display()


In [15]:
# Solve
gg.optimize()

Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (linux64 - "Ubuntu 24.04.2 LTS")

CPU model: AMD EPYC 7571, instruction set [SSE2|AVX|AVX2]
Thread count: 2 physical cores, 4 logical processors, using up to 4 threads

Optimize a model with 6 rows, 9 columns and 18 nonzeros
Model fingerprint: 0x44c3f6e8
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [4e+02, 2e+03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [8e+00, 3e+01]
Presolve time: 0.01s
Presolved: 6 rows, 9 columns, 18 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   3.200000e+01   0.000000e+00      0s
       4    1.4886000e+04   0.000000e+00   0.000000e+00      0s

Solved in 4 iterations and 0.01 seconds (0.00 work units)
Optimal objective  1.488600000e+04


In [16]:
# Print answer
print(f'Total Cost = ${gg.ObjVal:,.2f}')
for v in gg.getVars():
    if v.X > 0:
        print(f'{v.VarName} = {v.X}')

Total Cost = $14,886.00
flow[Bethesda,Boston] = 8.0
flow[Bethesda,Chicago] = 2.0
flow[Memphis,Atlanta] = 10.0
flow[Memphis,Chicago] = 12.0


## Culinary Compacts

Assume the two supply locations are already up and running so you do not need to incorporate the fixed costs.

In [ ]:
# Culinary Compacts
# Get data
# Culinary Compacts demand is in a sheet called 'cc_demand'


In [ ]:
# Convert demand to a dictionary


In [ ]:
# Create the markets as a list


In [ ]:
# Get data
# the capacity, variable, and fixed costs are in a sheet labeled 'gg_costs_and_cap'


In [ ]:
# Create the plants as a list


In [ ]:
# Create a dictionary to hold the capacity for each plant


In [ ]:
# Create a dictionary to hold the fixed cost for each plant


In [ ]:
# Get data
# Transportation costs are in our cc_costs_and_cap DataFrame


In [ ]:
# Convert costs to a list of lists


In [ ]:
# Create the model instance and set to minimize


# Create the flow variables from plants to markets


# Make sure you meet demand for each market


# Make sure stay under copacity at plants


# Update and display


In [ ]:
# solve


In [ ]:
# Print answer


## Combined Company - Culinary Gizmos

Now we need to incorporate the fixed costs for each manufacturing plant when deciding the best design configuration.

In [ ]:
# Culinary Gizmos - the combined company
# Get data
# Culinary Gizmos demand is in a sheet called 'combined_demand'


# Convert demand to a dictionary


# Create the markets as a list


In [ ]:
# Get data
# the capacity, variable, and fixed costs are in a sheet labeled 'combined_costs_and_cap'


In [ ]:
# Create the plants as a list


# Create a dictionary to hold the capacity for each plant


# Create a dictionary to hold the fixed cost for each plant


# Get data
# Transportation costs are in our all_costs_and_cap DataFrame


# Convert costs to a list of lists


In [ ]:
# Create the model instance and set to minimize


# Create the flow variables from plants to markets


# Create the binary variables to open plants


# update


# Make sure you meet demand for each market


# Make sure stay under copacity at plants


# Update and display


In [ ]:
# solve


In [ ]:
# Print answer
